# 04 — PyCaret-vergelijking

AutoML vergelijkt dezelfde vier operationele basismodellen. Geen random holdout; geen nieuwe finale modelkeuze.

## Bijdragen — Messi10

**Raoul Buffels:** dataverwerking en cleaning, EDA en hypothesen, features, modellen,
tuning, evaluatie en documentatie; Citi Bike-frontend en backend.
**Jorre Van Dyck:** hulp bij deployment op Render.
Codex hielp met implementatie, uitleg, tests en uitvoering. Ieder teamlid moet de
eigen keuzes mondeling kunnen uitleggen.

We bepalen de projectmap en laden de kleine tabellen. Grote bronbestanden blijven op schijf.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
locations=[Path.cwd(),Path.cwd().parent,Path.cwd()/'Citybike_full']
ROOT=next(p for p in locations if (p/'citibike').exists())
sys.path.insert(0,str(ROOT))
from citibike.config import REPORTS, DATA, MODELS
def report(name):
    return json.loads((REPORTS/name).read_text(encoding='utf-8'))
def table(name):
    display(pd.read_csv(REPORTS/name,dtype={'start_station_id':'string','end_station_id':'string','station_id':'string'}))


## Modelinvoer en preprocessing

We laden exact de output van notebook 02: kalenderwaarden en historische tellingen. De vijf chronologische folds hebben ieder 28 latere validatiedagen. Alleen training mag preprocessing leren; de eindtest wordt niet geladen.

In [2]:
from time import perf_counter
import gc
from citibike.model_data import load_model_data
from citibike.features import FEATURES, CALENDAR, SavedChronologicalCV
from citibike.modeling import OperationalRegressor, SeasonalNaive, metrics
from citibike.reproduction import save_model_reproduction
training, validation, prepared=load_model_data()
X_train, y_train=training[FEATURES],training.rides
X_validation, y_validation=validation[FEATURES],validation.rides
display(pd.DataFrame(prepared['folds']))
display(pd.DataFrame({'feature':FEATURES}))
print(f'Training: {len(training):,} rijen; validatie: {len(validation):,}. Geen testdata.')


,train_start,train_stop,validation_start,validation_stop,train_last_day,validation_first_day,validation_last_day
0,0,109512,109512,110184,2025-12-11,2025-12-12,2026-01-08
1,0,110184,110184,110856,2026-01-08,2026-01-09,2026-02-05
2,0,110856,110856,111528,2026-02-05,2026-02-06,2026-03-05
3,0,111528,111528,112200,2026-03-05,2026-03-06,2026-04-02
4,0,112200,112200,112872,2026-04-02,2026-04-03,2026-04-30


,feature
0,hour_sin
1,hour_cos
2,weekday_sin
3,weekday_cos
4,year_sin
5,year_cos
6,is_weekend
7,clock_hours
8,lag_1d
9,lag_2d


Training: 112,872 rijen; validatie: 1,464. Geen testdata.


## Setup

PyCaret test_data ontvangt hier mei–juni-validatie, niet de eindtest. preprocess=False voorkomt globale preprocessing. Ridge schaalt binnen zijn eigen fold-estimator.

In [3]:
from pycaret.regression import RegressionExperiment
from citibike.modeling import candidates
from citibike.reproduction import save_pycaret_reproduction
experiment=RegressionExperiment()
experiment.setup(data=training[FEATURES+['rides']],test_data=validation[FEATURES+['rides']],target='rides',fold_strategy=SavedChronologicalCV(prepared['folds']),fold=len(prepared['folds']),data_split_shuffle=False,fold_shuffle=False,preprocess=False,session_id=42,n_jobs=2,html=False,verbose=False,system_log=False,log_experiment=False,index=False)

## Werkelijke training

compare_models voert de vijf chronologische folds uit en fit de vier kandidaten. We controleren de werkelijk gebruikte interne rijen en bewaren deze reproductie apart.

In [4]:
ranked=experiment.compare_models(include=list(candidates().values()),sort='RMSE',n_select=4,errors='raise',verbose=False)
automl=save_pycaret_reproduction(experiment,ranked,training,validation,prepared)
display(automl)

,candidate,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
3,lightgbm,Light Gradient Boosting Machine,919.4277,2.219320e+06,1433.3589,0.7183,0.8604,1.0487,0.470
2,random_forest,Random Forest Regressor,917.6491,2.280610e+06,1460.7833,0.7014,0.8548,1.1461,23.482
1,ridge,Ridge Regression,1025.6297,2.668287e+06,1594.3144,0.6369,0.9592,1.1846,0.054
0,seasonal_naive,SeasonalNaive,1349.9824,4.550459e+06,2086.6729,0.3707,1.1928,1.2417,0.500


## Prediction en evaluatie

De teruggegeven estimators zijn op training gefit. We voorspellen expliciet op training en validatie en tonen RMSE, MAE en R².

In [5]:
rows=[]
for name,fitted in zip(automl.candidate,ranked):
    train_prediction=fitted.predict(X_train.astype('float32'))
    valid_prediction=fitted.predict(X_validation.astype('float32'))
    rows.append({'name':name,**metrics(y_validation,valid_prediction),**{'train_'+k:v for k,v in metrics(y_train,train_prediction).items()}})
display(pd.DataFrame(rows))
from citibike.config import atomic_json
atomic_json(REPORTS/'notebook_reproduction/pycaret_validation.json',rows)

,name,RMSE,MAE,R2,train_RMSE,train_MAE,train_R2
0,lightgbm,1956.952952,1158.502503,0.840851,778.365740,433.032423,0.928690
1,random_forest,1945.709748,1118.646342,0.842674,684.188257,359.153829,0.944902
2,ridge,1982.439599,1248.057527,0.836678,1021.691354,578.390271,0.877136
3,seasonal_naive,2826.164920,1510.127732,0.668076,1309.747668,677.011659,0.798089


## Conclusie en tuning

PyCaret verkleint numerieke dtypes; vooral LightGBM-binning kan daardoor verschillen. De oorspronkelijke handmatige validatiemetrics bepaalden de finale keuze. Tuning staat bij RF en LightGBM; AutoML krijgt geen nieuwe zoekprocedure.

In [6]:
manual=pd.DataFrame(report('model_comparison.json'))[['name','cv_RMSE']]
precision=automl[['candidate','RMSE']].merge(manual,left_on='candidate',right_on='name')
precision['CV_RMSE_verschil']=precision.RMSE-precision.cv_RMSE
display(precision)
display(pd.DataFrame(prepared['folds']))

,candidate,RMSE,name,cv_RMSE,CV_RMSE_verschil
0,lightgbm,1433.3589,lightgbm,1423.037130,10.321770
1,random_forest,1460.7833,random_forest,1460.783347,-0.000047
2,ridge,1594.3144,ridge,1594.314667,-0.000267
3,seasonal_naive,2086.6729,seasonal_naive,2086.672928,-0.000028


,train_start,train_stop,validation_start,validation_stop,train_last_day,validation_first_day,validation_last_day
0,0,109512,109512,110184,2025-12-11,2025-12-12,2026-01-08
1,0,110184,110184,110856,2026-01-08,2026-01-09,2026-02-05
2,0,110856,110856,111528,2026-02-05,2026-02-06,2026-03-05
3,0,111528,111528,112200,2026-03-05,2026-03-06,2026-04-02
4,0,112200,112200,112872,2026-04-02,2026-04-03,2026-04-30
